# 04 — Export ONNX & Benchmark latency

Giai đoạn 7 (`docs/PLAN.md`) — spec: `docs/specs/g7-export-backend.md`.

Mục tiêu: export `yolov8n_v1_baseline` (config ON, Giai đoạn 3 -- Giai đoạn 5
kết luận không cần retrain nên đây vẫn là model tốt nhất) sang ONNX, xác
nhận chạy được, benchmark latency PyTorch `.pt` vs ONNX Runtime `.onnx`.

**Chạy trên Colab** (cần GPU — Runtime → Change runtime type → T4 GPU).


## Setup — mount Drive + cd vào repo


In [ ]:
import os

REPO_DIR_NAME = "computer-vision-project"  # đổi nếu bạn git clone ra tên thư mục khác

try:
    from google.colab import drive

    drive.mount("/content/drive")
    drive_path = f"/content/drive/MyDrive/{REPO_DIR_NAME}"
    if not os.path.isdir(drive_path):
        raise FileNotFoundError(
            f"{drive_path} không tồn tại — kiểm tra lại bạn đã `git clone` repo vào "
            "đúng chỗ trong Drive chưa (T0.4), hoặc sửa REPO_DIR_NAME ở trên cho khớp."
        )
    os.chdir(drive_path)
except ImportError:
    pass  # không chạy trên Colab (vd Jupyter local) — giả định cwd đã là repo root

print("cwd:", os.getcwd())
assert os.path.isdir("scripts") and os.path.isdir("data"), (
    "Chưa đứng ở repo root — không thấy scripts/ và data/ ở cwd hiện tại."
)
BEST_WEIGHTS = "runs/detect/yolov8n_v1_baseline/weights/best.pt"
assert os.path.isfile(BEST_WEIGHTS), (
    f"{BEST_WEIGHTS} chưa có — chạy notebooks/02_train_detector.ipynb "
    "(Giai đoạn 2/3) trước để có checkpoint baseline."
)


## Đồng bộ code mới nhất


In [ ]:
assert os.path.isdir(".git"), (
    "cwd hiện tại không phải repo root (không thấy .git) — runtime Colab có "
    "thể vừa bị reset. Chạy lại cell 'Setup — mount Drive + cd vào repo' ở "
    "trên (mount + cd) trước, rồi chạy lại cell này."
)
!git checkout -- . && git pull


## Cài dependencies


In [ ]:
!pip install -q -r requirements.txt
import onnxruntime
import torch
import ultralytics

print("ultralytics", ultralytics.__version__, "| torch", torch.__version__)
print("onnxruntime", onnxruntime.__version__)
print("CUDA available:", torch.cuda.is_available())


## T7.1 — Export ONNX

`export_onnx()` (`src/models/export.py`) là wrapper mỏng quanh
`YOLO(...).export(format="onnx")` — không truyền `nms=True`, vì
`src/models/onnx_inference.py` tự làm NMS theo từng lớp (xem
`docs/specs/g7-export-backend.md` — 2 học viên khác tư thế có thể chồng
box thật, NMS gộp lớp có thể xoá nhầm).


In [ ]:
from src.models.export import export_onnx

onnx_path = export_onnx(BEST_WEIGHTS, imgsz=640)
print("Exported to:", onnx_path)

import os as _os
assert _os.path.isfile(onnx_path), f"{onnx_path} không tồn tại sau khi export."


Xác nhận `onnxruntime` load được + chạy thử 1 lần không lỗi (dùng lại
`run_onnx_detection()` — đúng code path `app/service.py` sẽ chạy thật, xem
`src/models/onnx_inference.py`).


In [ ]:
import cv2
import onnxruntime

from src.models.onnx_inference import run_onnx_detection

CLASS_NAMES = ["bridge", "downward", "plank", "shoulderstand", "tree"]
CONF_THRESHOLD = 0.5

session = onnxruntime.InferenceSession(onnx_path, providers=["CPUExecutionProvider"])

# 1 ảnh test bất kỳ để xác nhận pipeline chạy hết không lỗi.
sample_image_path = next(iter(sorted((__import__("pathlib").Path("data/raw/yoga_v1/test/images")).glob("*.jpg"))))
sample_image_bgr = cv2.imread(str(sample_image_path))
sample_image_rgb = cv2.cvtColor(sample_image_bgr, cv2.COLOR_BGR2RGB)

detections = run_onnx_detection(session, sample_image_rgb, CLASS_NAMES, CONF_THRESHOLD)
print(f"Ảnh test: {sample_image_path.name}")
print(f"{len(detections)} detection(s):")
for d in detections:
    print(f"  {d.class_name} conf={d.confidence:.2f} box={[round(v, 1) for v in d.box]}")


## T7.2 — Benchmark latency: PyTorch `.pt` vs ONNX Runtime `.onnx`

N=50 lần inference trên cùng 1 ảnh test, đo trên GPU T4 Colab — **không
đại diện cho CPU thật lúc deploy** (Giai đoạn 9 dùng CPU free-tier), chỉ để
so sánh tương đối PyTorch vs ONNX. ONNX dùng đúng `run_onnx_detection()`
(cùng code path `app/service.py`), PyTorch dùng `YOLO(...).predict()`.


In [ ]:
import time

from ultralytics import YOLO

N = 50
pt_model = YOLO(BEST_WEIGHTS)

# Warmup (lần đầu luôn chậm hơn do lazy init/CUDA context) — không tính vào đo.
pt_model.predict(source=str(sample_image_path), verbose=False)
run_onnx_detection(session, sample_image_rgb, CLASS_NAMES, CONF_THRESHOLD)

pt_times = []
for _ in range(N):
    t0 = time.perf_counter()
    pt_model.predict(source=str(sample_image_path), verbose=False)
    pt_times.append((time.perf_counter() - t0) * 1000)

onnx_times = []
for _ in range(N):
    t0 = time.perf_counter()
    run_onnx_detection(session, sample_image_rgb, CLASS_NAMES, CONF_THRESHOLD)
    onnx_times.append((time.perf_counter() - t0) * 1000)

import pandas as pd

def summarize(name, times):
    avg_ms = sum(times) / len(times)
    return {"backend": name, "avg_latency_ms": avg_ms, "fps": 1000 / avg_ms}

benchmark = pd.DataFrame([summarize("PyTorch (.pt)", pt_times), summarize("ONNX Runtime (.onnx)", onnx_times)]).set_index("backend")
benchmark


**Kết quả (chạy thật trên Colab, GPU T4, N=50):**

| Backend | avg latency (ms) | FPS |
|---|---|---|
| PyTorch (.pt) | 247.94 | 4.03 |
| ONNX Runtime (.onnx) | 206.29 | 4.85 |

ONNX nhanh hơn PyTorch ~17% (206.29ms vs 247.94ms, ~4.85 vs ~4.03 FPS) —
đúng như kỳ vọng (ONNX Runtime tối ưu graph, bớt overhead Python so với
`ultralytics.YOLO.predict()`). Cả 2 số đều **thấp** so với FPS video mượt
(~30) vì đo trên GPU T4 dùng chung của Colab free-tier — không đại diện
cho CPU thật lúc deploy (Giai đoạn 9 dùng Hugging Face Spaces CPU
free-tier), chỉ để so sánh tương đối 2 backend. Latency thật trên CPU
deploy sẽ đo lại khi có server thật (không suy diễn từ số GPU này).

## Copy `best.onnx` sang Drive

Cell Setup đầu notebook đã `cd` vào repo trong Drive — `onnx_path` (cùng
thư mục `runs/detect/yolov8n_v1_baseline/weights/`) tự động đã nằm trong
Drive, không cần copy tay. Tải file này về máy local (`models/best.onnx`,
hoặc set env var `MODEL_PATH`) để chạy `app/` thật — xem
`docs/specs/g7-export-backend.md` mục "Cách bạn tự test".
